# Simple Multi-GPU: Stable Diffusion LoRA

Teach Stable Diffusion a new style or concept using LoRA.

- **Model:** Stable Diffusion 1.5 (860M UNet — the part we train)
- **Method:** LoRA on the UNet attention layers
- **Data:** A few images of one concept (e.g., a style, an object)
- **Platform:** Kaggle 2x T4 (free) or Colab T4 (free)

### How Stable Diffusion works (30 seconds)

```
"a photo of a cat" ──> Text Encoder ──> text embedding
                                              |
Pure noise ──> UNet (denoises step by step) <─┘
                    |
               Clean image
```

The **UNet** is the brain. It learns to remove noise from images,
guided by text. We fine-tune only the UNet's attention layers with LoRA.

### What LoRA does here

Instead of updating all 860M UNet parameters, LoRA adds tiny
trainable matrices (~1MB) to the attention layers. Same result, 100x less memory.

## Step 1: Install

In [ ]:
!pip install -q diffusers transformers accelerate peft datasets

## Step 2: Detect GPUs

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

## Step 3: Write Accelerate Config

No DeepSpeed this time — diffusion training doesn't need optimizer offloading.
We just use Accelerate's built-in multi-GPU (DistributedDataParallel).
This is simpler and faster for models that fit in GPU memory.

In [ ]:
# Simple multi-GPU config — no DeepSpeed needed
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Accelerate config: {NUM_GPUS} GPU(s), fp16, multi-GPU mode")

## Step 4: Download Training Images

We'll use a small dataset of images. You can swap this for your own images.

For this demo we use the `lambdalabs/naruto-blip-captions` dataset —
anime-style character images with text captions. The model will learn
to generate images in this style.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("lambdalabs/naruto-blip-captions", split="train")
print(f"Dataset: {len(dataset)} images with captions")
print(f"Example caption: {dataset[0]['text']}")

# Show a few examples
from IPython.display import display
for i in range(3):
    print(f"\n{dataset[i]['text']}")
    display(dataset[i]["image"].resize((256, 256)))

## Step 5: Write Training Script

This is the full training loop. Every step is commented.

**What happens each training step:**
1. Load a clean image
2. Add random noise to it
3. Ask the UNet: "given this noisy image + text caption, predict the noise"
4. Compare predicted noise vs actual noise → that's the loss
5. Update LoRA weights to make the prediction better

In [ ]:
%%writefile train_diffusion.py
"""Distributed Stable Diffusion LoRA training."""
import torch
import torch.nn.functional as F
import os, json, time, math

from accelerate import Accelerator
from datasets import load_dataset
from diffusers import (
    AutoencoderKL,        # VAE — compresses images to latent space
    DDPMScheduler,        # Noise scheduler — controls how noise is added/removed
    UNet2DConditionModel, # UNet — the denoiser we're training
)
from diffusers.loaders import LoraLoaderMixin
from transformers import CLIPTextModel, CLIPTokenizer  # Text encoder
from peft import LoraConfig, get_peft_model
from torchvision import transforms
from torch.utils.data import DataLoader

# ── Config ──────────────────────────────────────────────
MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
OUTPUT_DIR = "./sd_lora_output"
RESOLUTION = 512          # Image size (SD 1.5 native)
BATCH_SIZE = 1            # Per GPU (images are large)
GRAD_ACCUM = 4            # Effective batch = 1 * 4 * num_gpus
LR = 1e-4
NUM_EPOCHS = 5
LORA_RANK = 4             # Small rank = tiny LoRA, fast training

# ── Step 1: Set up Accelerator ──────────────────────────
# Accelerator handles: multi-GPU, mixed precision, gradient accumulation
accelerator = Accelerator(
    gradient_accumulation_steps=GRAD_ACCUM,
    mixed_precision="fp16",
)

# ── Step 2: Load model components ───────────────────────
# Stable Diffusion has 4 parts:
#   1. Text encoder (CLIP) — converts text to embeddings     [FROZEN]
#   2. VAE encoder         — compresses image to latent       [FROZEN]
#   3. UNet                — denoises latents guided by text  [TRAIN with LoRA]
#   4. VAE decoder         — decompresses latent to image     [FROZEN]

tokenizer = CLIPTokenizer.from_pretrained(MODEL, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float16)
unet = UNet2DConditionModel.from_pretrained(MODEL, subfolder="unet", torch_dtype=torch.float16)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL, subfolder="scheduler")

# Freeze everything except UNet
text_encoder.requires_grad_(False)
vae.requires_grad_(False)

# ── Step 3: Add LoRA to UNet ────────────────────────────
# Only the cross-attention layers (where text meets image) get LoRA
unet = get_peft_model(unet, LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,  # alpha = rank is a common default
    target_modules=["to_q", "to_v", "to_k", "to_out.0"],  # Attention projections
    lora_dropout=0.0,
))

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
unet.print_trainable_parameters()

# ── Step 4: Load dataset ────────────────────────────────
dataset = load_dataset("lambdalabs/naruto-blip-captions", split="train")
print(f"Dataset: {len(dataset)} images")

# Image preprocessing — resize, normalize to [-1, 1] (what SD expects)
image_transforms = transforms.Compose([
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),           # [0, 1]
    transforms.Normalize([0.5], [0.5]),  # [-1, 1]
])

def collate_fn(examples):
    """Prepare a batch: tokenize captions, transform images."""
    images = [image_transforms(ex["image"].convert("RGB")) for ex in examples]
    captions = [ex["text"] for ex in examples]

    tokens = tokenizer(
        captions, padding="max_length", truncation=True,
        max_length=tokenizer.model_max_length, return_tensors="pt",
    )
    return {
        "pixel_values": torch.stack(images),
        "input_ids": tokens.input_ids,
    }

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# ── Step 5: Optimizer ───────────────────────────────────
# Only LoRA parameters are updated
optimizer = torch.optim.AdamW(unet.parameters(), lr=LR)

# ── Step 6: Prepare for distributed training ────────────
# accelerator.prepare() wraps everything for multi-GPU
unet, optimizer, dataloader = accelerator.prepare(unet, optimizer, dataloader)

# Move frozen models to GPU (they don't need gradients)
text_encoder.to(accelerator.device)
vae.to(accelerator.device)

# ── Step 7: Training loop ───────────────────────────────
loss_history = []
global_step = 0
num_batches = len(dataloader)
total_steps = NUM_EPOCHS * math.ceil(num_batches / GRAD_ACCUM)

torch.cuda.reset_peak_memory_stats()
train_start = time.time()
step_times = []

if accelerator.is_main_process:
    print(f"\nTraining for {NUM_EPOCHS} epochs, ~{total_steps} optimizer steps")
    print(f"Batch: {BATCH_SIZE}/gpu x {GRAD_ACCUM} accum x {accelerator.num_processes} GPU = {BATCH_SIZE * GRAD_ACCUM * accelerator.num_processes} effective")

for epoch in range(NUM_EPOCHS):
    unet.train()
    epoch_loss = 0.0
    num_steps_in_epoch = 0

    for batch in dataloader:
        with accelerator.accumulate(unet):
            # ── What happens each step ──

            # 1. Encode image to latent space using VAE
            #    Image (3x512x512) → Latent (4x64x64) — 8x smaller
            with torch.no_grad():
                latents = vae.encode(batch["pixel_values"].half()).latent_dist.sample()
                latents = latents * vae.config.scaling_factor  # Scale to match UNet's expected range

            # 2. Add random noise to the latents
            #    Pick a random timestep (how much noise to add)
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                     (latents.shape[0],), device=latents.device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # 3. Get text embeddings from CLIP
            with torch.no_grad():
                text_embeds = text_encoder(batch["input_ids"])[0]

            # 4. UNet predicts the noise that was added
            noise_pred = unet(noisy_latents, timesteps, text_embeds).sample

            # 5. Loss = how wrong was the noise prediction?
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")

            # 6. Backprop + update LoRA weights
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.detach().item()
        num_steps_in_epoch += 1

        # Log every accumulation cycle
        if accelerator.sync_gradients:
            global_step += 1
            step_times.append(time.time())
            avg_loss = epoch_loss / num_steps_in_epoch
            loss_history.append({"step": global_step, "loss": round(avg_loss, 4), "epoch": epoch + 1})

            if accelerator.is_main_process and global_step % 5 == 0:
                print(f"  Step {global_step}/{total_steps} | Loss: {avg_loss:.4f}")

    if accelerator.is_main_process:
        avg = epoch_loss / max(num_steps_in_epoch, 1)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} done | Avg loss: {avg:.4f}")

train_time = time.time() - train_start

# ── Step 8: Save LoRA weights ───────────────────────────
if accelerator.is_main_process:
    # Unwrap from accelerator wrapper, then save just the LoRA weights
    unwrapped = accelerator.unwrap_model(unet)
    unwrapped.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"\nLoRA weights saved to {OUTPUT_DIR}")

    # Save metrics
    gpu_mem_alloc = torch.cuda.max_memory_allocated() / 1e9
    gpu_mem_res = torch.cuda.max_memory_reserved() / 1e9
    gpu_total = torch.cuda.get_device_properties(0).total_mem / 1e9

    if len(step_times) > 2:
        durations = [step_times[i+1] - step_times[i] for i in range(len(step_times)-1)]
        avg_step = sum(durations) / len(durations)
    else:
        avg_step = train_time / max(global_step, 1)

    metrics = {
        "model": MODEL,
        "task": "text-to-image LoRA",
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_pct": round(trainable_params / total_params * 100, 2),
        "lora_rank": LORA_RANK,
        "num_gpus": accelerator.num_processes,
        "gpu_name": torch.cuda.get_device_name(0),
        "gpu_mem_allocated_gb": round(gpu_mem_alloc, 2),
        "gpu_mem_reserved_gb": round(gpu_mem_res, 2),
        "gpu_mem_total_gb": round(gpu_total, 1),
        "train_time_sec": round(train_time, 1),
        "total_steps": global_step,
        "total_images": global_step * BATCH_SIZE * GRAD_ACCUM * accelerator.num_processes,
        "images_per_sec": round(global_step * BATCH_SIZE * GRAD_ACCUM * accelerator.num_processes / train_time, 2),
        "avg_step_time_sec": round(avg_step, 3),
        "final_loss": loss_history[-1]["loss"] if loss_history else None,
        "loss_history": loss_history,
        "batch_size_per_gpu": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM * accelerator.num_processes,
        "learning_rate": LR,
        "num_epochs": NUM_EPOCHS,
        "resolution": RESOLUTION,
    }
    with open("training_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("Metrics saved to training_metrics.json")
    print(f"Done! {train_time:.0f}s")

## Step 6: Launch Training

In [ ]:
print(f"Launching on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train_diffusion.py

elapsed = time.time() - start
print(f"\nTotal time: {elapsed:.0f}s")

## Step 7: Performance Dashboard

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as ticker
from IPython.display import HTML, display

with open("training_metrics.json") as f:
    m = json.load(f)

# --- Loss Curve ---
fig, ax1 = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#0d1117")
ax1.set_facecolor("#0d1117")

steps = [h["step"] for h in m["loss_history"]]
losses = [h["loss"] for h in m["loss_history"]]

color_loss = "#58a6ff"
ax1.plot(steps, losses, color=color_loss, linewidth=2, label="MSE Loss", zorder=3)
ax1.fill_between(steps, losses, alpha=0.1, color=color_loss)
ax1.set_xlabel("Step", color="#8b949e", fontsize=11)
ax1.set_ylabel("Noise Prediction Loss (MSE)", color=color_loss, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color_loss)
ax1.tick_params(axis="x", colors="#8b949e")
ax1.grid(True, alpha=0.15, color="#30363d")
ax1.spines["top"].set_visible(False)
for spine in ax1.spines.values():
    spine.set_color("#30363d")

# Mark epoch boundaries
epochs_seen = set()
for h in m["loss_history"]:
    if h["epoch"] not in epochs_seen:
        epochs_seen.add(h["epoch"])
        if h["epoch"] > 1:
            ax1.axvline(x=h["step"], color="#f0883e", alpha=0.3, linestyle="--")

fig.suptitle("Diffusion Training — Noise Prediction Loss", color="#e6edf3", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# --- GPU Memory ---
fig2, ax3 = plt.subplots(figsize=(6, 3))
fig2.patch.set_facecolor("#0d1117")
ax3.set_facecolor("#0d1117")
mem_labels = ["Allocated", "Reserved", "Total"]
mem_vals = [m["gpu_mem_allocated_gb"], m["gpu_mem_reserved_gb"], m["gpu_mem_total_gb"]]
colors = ["#3fb950", "#58a6ff", "#30363d"]
bars = ax3.barh(mem_labels, mem_vals, color=colors, height=0.5, edgecolor="#0d1117")
for bar, val in zip(bars, mem_vals):
    ax3.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f} GB",
             va="center", color="#e6edf3", fontsize=11, fontweight="bold")
ax3.set_xlim(0, m["gpu_mem_total_gb"] * 1.3)
ax3.set_title(f"GPU Memory — {m['gpu_name']}", color="#e6edf3", fontsize=13, fontweight="bold")
ax3.tick_params(colors="#8b949e")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
for spine in ax3.spines.values():
    spine.set_color("#30363d")
plt.tight_layout()
plt.show()

# --- HTML Dashboard ---
loss_drop = ""
if len(losses) >= 2:
    pct = (losses[0] - losses[-1]) / losses[0] * 100
    loss_drop = f"{pct:.0f}% drop"

mem_util = m["gpu_mem_allocated_gb"] / m["gpu_mem_total_gb"] * 100

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 780px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Throughput</div>
      <div style="color: #58a6ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['images_per_sec']:.1f}</div>
      <div style="color: #8b949e; font-size: 12px;">images/sec</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Images Trained</div>
      <div style="color: #3fb950; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['total_images']}</div>
      <div style="color: #8b949e; font-size: 12px;">{m['num_epochs']} epochs</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Final Loss</div>
      <div style="color: #f0883e; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['final_loss']:.4f}</div>
      <div style="color: #8b949e; font-size: 12px;">{loss_drop}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Train Time</div>
      <div style="color: #d2a8ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['train_time_sec']:.0f}s</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_steps']} steps</div>
    </div>
  </div>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">Model & Training</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">Model</td><td style="text-align:right; font-weight:600;">SD 1.5 UNet</td></tr>
        <tr><td style="color:#8b949e;">UNet params</td><td style="text-align:right;">{m['total_params']/1e6:.0f}M</td></tr>
        <tr><td style="color:#8b949e;">LoRA (rank {m['lora_rank']})</td>
            <td style="text-align:right; color:#3fb950;">{m['trainable_params']/1e3:.0f}K ({m['trainable_pct']:.2f}%)</td></tr>
        <tr><td style="color:#8b949e;">Effective batch</td>
            <td style="text-align:right;">{m['batch_size_per_gpu']} x {m['grad_accum_steps']} x {m['num_gpus']}GPU = {m['effective_batch_size']}</td></tr>
        <tr><td style="color:#8b949e;">Resolution</td><td style="text-align:right;">{m['resolution']}x{m['resolution']}</td></tr>
        <tr><td style="color:#8b949e;">Precision</td><td style="text-align:right;">fp16</td></tr>
      </table>
    </div>
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">GPU & Memory</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">GPUs</td><td style="text-align:right; font-weight:600;">{m['num_gpus']}x {m['gpu_name']}</td></tr>
        <tr><td style="color:#8b949e;">VRAM total</td><td style="text-align:right;">{m['gpu_mem_total_gb']} GB per GPU</td></tr>
        <tr><td style="color:#8b949e;">Peak allocated</td>
            <td style="text-align:right; color:#3fb950;">{m['gpu_mem_allocated_gb']} GB ({mem_util:.0f}%)</td></tr>
        <tr><td style="color:#8b949e;">Peak reserved</td><td style="text-align:right;">{m['gpu_mem_reserved_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Avg step time</td><td style="text-align:right;">{m['avg_step_time_sec']*1000:.0f} ms</td></tr>
        <tr><td style="color:#8b949e;">Backend</td><td style="text-align:right;">Multi-GPU (DDP)</td></tr>
      </table>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 14px 18px;
              margin-top: 12px; display: flex; justify-content: space-between; align-items: center;">
    <span style="color: #8b949e; font-size: 12px;">GPU Memory Utilization</span>
    <div style="flex: 1; margin: 0 16px; background: #21262d; border-radius: 6px; height: 18px; overflow: hidden;">
      <div style="width: {mem_util:.0f}%; height: 100%; border-radius: 6px;
                  background: linear-gradient(90deg, #238636, #3fb950);"></div>
    </div>
    <span style="color: #3fb950; font-size: 13px; font-weight: 700;">{mem_util:.0f}%</span>
  </div>

</div>
"""
display(HTML(html))

## Step 8: Generate Images with Your LoRA

Load the base SD 1.5 model, attach your LoRA weights, and generate.

In [ ]:
from diffusers import StableDiffusionPipeline
from IPython.display import display
import torch

# Load base pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

# Load your LoRA weights on top
pipe.load_lora_weights("./sd_lora_output")
print("LoRA loaded!")

# Generate
prompts = [
    "a naruto character with red hair and blue eyes, anime style",
    "a ninja standing on a mountain at sunset, anime style",
    "a warrior with a glowing sword in a dark forest, anime style",
]

for prompt in prompts:
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
    print(f"\n\"{prompt}\"")
    display(image)

In [ ]:
# Side-by-side: with LoRA vs without
prompt = "a ninja warrior in anime style"

# With LoRA (already loaded)
img_lora = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=torch.manual_seed(42)).images[0]

# Without LoRA
pipe.unload_lora_weights()
img_base = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=torch.manual_seed(42)).images[0]

# Show side by side
from PIL import Image
combined = Image.new("RGB", (img_base.width * 2 + 20, img_base.height), "black")
combined.paste(img_base, (0, 0))
combined.paste(img_lora, (img_base.width + 20, 0))

print(f"\"{prompt}\"")
print("Left: Base SD 1.5  |  Right: + Your LoRA")
display(combined)

---

## How It All Works

### The Training Loop (what happens each step)

```
                        "a ninja with red hair"
                                |
                          CLIP Text Encoder  [frozen]
                                |
                          text embeddings
                                |
Clean Image ──> VAE Encode ──> latent ──> Add Noise ──> noisy latent
   [frozen]     [frozen]                                     |
                                                        UNet + LoRA
                                                     "predict the noise"
                                                             |
                                                      predicted noise
                                                             |
                                   Loss = MSE(predicted noise, actual noise)
                                                             |
                                                  Update LoRA weights only
```

### What's frozen vs trained

| Component | Params | Trained? | Why |
|-----------|--------|----------|-----|
| CLIP Text Encoder | 123M | Frozen | Already understands language |
| VAE (encoder+decoder) | 83M | Frozen | Already compresses images well |
| UNet (base) | 860M | Frozen | Base denoising ability preserved |
| UNet LoRA | ~400K | **Trained** | Learns your new style/concept |

### Why no DeepSpeed?

The text/audio notebooks used DeepSpeed ZeRO-2 with CPU offloading.
Here we don't need it because:
- Only LoRA params (~400K) have gradients — tiny optimizer state
- The frozen components (VAE, CLIP) just do inference
- Plain multi-GPU (DDP) is simpler and has less overhead

### Generation (after training)

```
"a ninja at sunset" ──> CLIP ──> text embeddings
                                       |
Random noise ──> UNet+LoRA (denoise 30 steps) ──> clean latent ──> VAE Decode ──> Image!
```

| Platform | GPUs | Cost |
|----------|------|------|
| **Kaggle** | 2x T4 | Free (30h/week) |
| **Colab** | 1x T4 | Free |